# Demo: `ask()` method — Dynamic Skill Selection & Token Budget

This notebook demonstrates the `ForecastingAssistant.ask()` method with the optimizations:

1. **Dynamic skill selection** — Skills are chosen based on `task_type` + keyword matching
2. **Skill conflict resolution** — Specialized skills suppress incompatible generic ones
3. **Token budget trimming** — Skills are pruned to fit a given token budget
4. **Context-aware prompting** — Pre-computed profiles/plans are serialized into context
5. **Results mode** — Ask questions about predictions/metrics from a `forecast()` run
6. **Dynamic Ollama context sizing** — `num_ctx` is estimated from prompt size
7. **Selective reference inclusion** — `include_reference=False` saves ~7600 tokens

**Requirements:** An Ollama instance running locally (or adjust `llm=` for another provider).

In [1]:
import sys
sys.path.insert(0, "..")

In [2]:
import warnings

import numpy as np
import pandas as pd

from skforecast_ai import ForecastingAssistant
from skforecast_ai.llm import (
    select_skills,
    estimate_prompt_tokens,
    load_skill
)
from skforecast_ai.llm.skills import ALL_SKILLS, _SKILL_TOKEN_ESTIMATES

warnings.filterwarnings("ignore")

## 1. Skill selection engine

Skills are selected in two phases:
- **Profile-based**: maps `task_type` → base skills
- **Keyword augmentation**: scans the question for topic patterns
- **Conflict resolution**: specialized skills suppress incompatible generic ones

In [3]:
# Single-series task with a question about prediction intervals
skills = select_skills(
    task_type="single_series",
    question="How do I get prediction intervals with conformal calibration?",
)
print("Selected skills:", skills)
print(f"Estimated tokens: {estimate_prompt_tokens(skills)}")

Selected skills: ['choosing-a-forecaster', 'forecasting-single-series', 'prediction-intervals']
Estimated tokens: 7459


In [4]:
# Multi-series + hyperparameter tuning keywords
skills = select_skills(
    task_type="multi_series",
    question="Run a bayesian search to optimize lags and n_estimators",
)
print("Selected skills:", skills)
print(f"Estimated tokens: {estimate_prompt_tokens(skills)}")

Selected skills: ['choosing-a-forecaster', 'forecasting-multiple-series', 'hyperparameter-optimization', 'autocorrelation-and-lag-selection']
Estimated tokens: 9914


In [5]:
# Foundation model question (no task_type context)
skills = select_skills(
    task_type=None,
    question="Can I use Chronos-2 for zero-shot forecasting?",
)
print("Selected skills:", skills)
print(f"Estimated tokens: {estimate_prompt_tokens(skills)}")

Selected skills: ['choosing-a-forecaster', 'foundation-forecasting']
Estimated tokens: 6221


In [6]:
# Conflict resolution: foundation keyword suppresses multi-series skill
# ForecasterFoundation handles multi-series natively (not ForecasterRecursiveMultiSeries)
skills = select_skills(
    task_type="multi_series",
    question="Can I use Chronos-2 for my multiple store series?",
)
print("Selected skills:", skills)
print("  → 'forecasting-multiple-series' is suppressed by 'foundation-forecasting'")
print(f"  Estimated tokens: {estimate_prompt_tokens(skills)}")


Selected skills: ['choosing-a-forecaster', 'foundation-forecasting']
  → 'forecasting-multiple-series' is suppressed by 'foundation-forecasting'
  Estimated tokens: 6221


In [7]:
# Without foundation keyword → multi-series skill remains
skills_no_conflict = select_skills(
    task_type="multi_series",
    question="How should I forecast multiple store series?",
)
print("No conflict:", skills_no_conflict)
print(f"  Estimated tokens: {estimate_prompt_tokens(skills_no_conflict)}")


No conflict: ['choosing-a-forecaster', 'forecasting-multiple-series']
  Estimated tokens: 4492


## 2. Token budget trimming

When a `token_budget` is specified, skills are included in order until the budget is exhausted.

In [8]:
# Show all skill sizes
print(f"{'Skill':<40} {'Tokens':>8}")
print("-" * 50)
for skill, tokens in sorted(_SKILL_TOKEN_ESTIMATES.items(), key=lambda x: x[1]):
    print(f"{skill:<40} {tokens:>8}")
print("-" * 50)
print(f"{'TOTAL':<40} {sum(_SKILL_TOKEN_ESTIMATES.values()):>8}")

Skill                                      Tokens
--------------------------------------------------
forecasting-single-series                    1224
drift-detection                              1249
feature-selection                            1407
forecasting-multiple-series                  1482
autocorrelation-and-lag-selection            2044
troubleshooting-common-errors                2108
choosing-a-forecaster                        2810
foundation-forecasting                       3211
prediction-intervals                         3225
hyperparameter-optimization                  3378
statistical-models                           3755
deep-learning-forecasting                    3883
metric-selection                             5072
feature-engineering                          8126
complete-api-reference                      11409
--------------------------------------------------
TOTAL                                       54383


In [9]:
# Question that matches many keywords but with a tight budget
skills_full = select_skills(
    task_type="single_series",
    question="I need ARIMA with prediction intervals and hyperparameter tuning",
    token_budget=None,  # No limit
)
print(f"No budget   -> {len(skills_full)} skills: {skills_full}")
print(f"  Estimated: {estimate_prompt_tokens(skills_full)} tokens\n")

skills_trimmed = select_skills(
    task_type="single_series",
    question="I need ARIMA with prediction intervals and hyperparameter tuning",
    token_budget=8000,  # Tight budget
)
print(f"Budget=8000 -> {len(skills_trimmed)} skills: {skills_trimmed}")
print(f"  Estimated: {estimate_prompt_tokens(skills_trimmed)} tokens")

No budget   -> 4 skills: ['choosing-a-forecaster', 'prediction-intervals', 'hyperparameter-optimization', 'statistical-models']
  Estimated: 13368 tokens

Budget=8000 -> 2 skills: ['choosing-a-forecaster', 'prediction-intervals']
  Estimated: 6235 tokens


## 3. Token estimation with/without API reference

The `include_reference` flag adds the full `llms-base.txt` (~7600 tokens). Use it only when the question needs broad API knowledge.

In [10]:
skills = ["choosing-a-forecaster", "forecasting-single-series"]

without_ref = estimate_prompt_tokens(skills, include_reference=False)
with_ref = estimate_prompt_tokens(skills, include_reference=True)

print(f"Without reference: {without_ref:,} tokens")
print(f"With reference:    {with_ref:,} tokens")
print(f"Savings:           {with_ref - without_ref:,} tokens ({100*(with_ref - without_ref)/with_ref:.0f}%)")

Without reference: 4,234 tokens
With reference:    11,834 tokens
Savings:           7,600 tokens (64%)


## 4. Full `ask()` — Q&A mode (no data)

Ask general questions about skforecast. Skills are auto-selected from keywords.

In [11]:
# assistant = ForecastingAssistant(llm="ollama:qwen2.5:7b-instruct")

In [23]:
API_KEY = "include_your_actual_key_here"  # Replace with your actual key

assistant = ForecastingAssistant(
    llm="google:gemini-3-flash-preview",
    api_key=API_KEY,
)

In [24]:
result = assistant.ask(
    prompt="What's the difference between ForecasterRecursive and ForecasterDirect?"
)
print(result.explanation)

[LLM unavailable] status_code: 400, model_name: gemini-3-flash-preview, body: {'error': {'code': 400, 'message': 'API key not valid. Please pass a valid API key.', 'status': 'INVALID_ARGUMENT', 'details': [{'@type': 'type.googleapis.com/google.rpc.ErrorInfo', 'reason': 'API_KEY_INVALID', 'domain': 'googleapis.com', 'metadata': {'service': 'generativelanguage.googleapis.com'}}, {'@type': 'type.googleapis.com/google.rpc.LocalizedMessage', 'locale': 'en-US', 'message': 'API key not valid. Please pass a valid API key.'}]}}


In [14]:
# Keyword triggers foundation-forecasting skill
result = assistant.ask(
    prompt="How do I use Chronos-2 for multi-series zero-shot forecasting?"
)
print(result.explanation)

To use Amazon Chronos-2 for multi-series zero-shot forecasting in skforecast, you use the `ForecasterFoundation` class with the `FoundationModel` adapter. In this mode, the model does not "train" on your data in the traditional sense; instead, it uses the provided history as context to generate predictions based on its pre-trained weights.

### Prerequisites

You must install the Chronos backend separately, as it is not included in the core skforecast installation:

```bash
pip install chronos-forecasting
```

### 1. Initialize the Model

You start by configuring the `FoundationModel`. For multi-series, you can enable `cross_learning=True`, which allows Chronos-2 to share information across the different series in your batch during the inference process.

```python
from skforecast.foundation import FoundationModel, ForecasterFoundation

# 1. Configure the model
model = FoundationModel(
    model_id='autogluon/chronos-2-small', # Or 'base', 'large'
    device_map='auto',                

## 5. Full `ask()` — Explain mode (with data)

When data is provided, the assistant runs the deterministic profiling + plan pipeline first, then explains the result via the LLM.

In [15]:
# Create sample data
rng = np.random.default_rng(42)
n = 365
dates = pd.date_range("2022-01-01", periods=n, freq="D")
target = 100 + np.cumsum(rng.normal(0, 1, n)) + 5 * np.sin(2 * np.pi * np.arange(n) / 7)

df = pd.DataFrame({
    "date": dates,
    "sales": target,
    "promo": rng.normal(50, 10, n),
})
df.head()

,date,sales,promo
0,2022-01-01,100.304717,64.814555
1,2022-01-02,103.173890,42.564118
2,2022-01-03,104.889824,41.777500
3,2022-01-04,103.125168,52.023062
4,2022-01-05,96.835295,58.443852


In [16]:
result = assistant.ask(
    prompt="Explain the forecasting strategy for this dataset",
    data=df,
    target="sales",
    date_column="date",
    steps=14,
)

print("=" * 60)
print("PROFILE")
print("=" * 60)
print(f"Task type: {result.profile.task_type}")
print(f"Forecaster: {result.profile.forecaster}")
print(f"Estimator: {result.profile.estimator}")

print("\n" + "=" * 60)
print("PLAN")
print("=" * 60)
print(f"Explanation: {result.plan.explanation}")

print("\n" + "=" * 60)
print("LLM EXPLANATION")
print("=" * 60)
print(result.explanation)

PROFILE
Task type: single_series
Forecaster: ForecasterRecursive
Estimator: LGBMRegressor

PLAN
Explanation: Plan: ForecasterRecursive + LGBMRegressor. Lags: [1, 3, 7, 8, 10, 12, 118]. Window features: ['mean(window=7)', 'mean(window=91)']. NaN rows kept (NaN-tolerant estimator). Exogenous variables included. MAE is interpretable, robust to outliers, and works at any scale.

LLM EXPLANATION
The forecasting strategy for your sales data is designed to capture both short-term fluctuations and long-term seasonal patterns using a machine learning approach. Based on your dataset of 365 daily observations, here is an explanation of the recommended plan.

### Core Forecaster Strategy
The plan utilizes **ForecasterRecursive** with a **LightGBM (LGBMRegressor)** estimator. 

*   **Recursive Strategy:** This model predicts one day at a time, using the prediction from the previous step as an input for the next. This is a standard and efficient approach for a 14-day horizon.
*   **Estimator Choice:

## 6. Explicit skill override

Override auto-selection to force specific skills (e.g., troubleshooting).

In [17]:
result = assistant.ask(
    prompt="I'm getting a ValueError when calling predict(). What should I check?",
    skills=["troubleshooting-common-errors", "complete-api-reference"],
)
print(result.explanation)

When calling `predict()` in skforecast, a `ValueError` usually stems from a mismatch between the data provided and the expectations of the trained model.

Here are the primary areas to check:

### 1. Index and Frequency
The most common cause of `ValueError` is an issue with the `DatetimeIndex`.
*   **Missing Frequency:** Ensure your training data had a frequency set (e.g., `'D'`, `'h'`, `'MS'`). Check this with `df.index.freq`.
*   **Gap in Time:** If you are predicting without providing a `last_window`, skforecast uses the data from the last training point. If there is a gap between the end of training and your prediction start, it may fail.
*   **Fix:** Use `data = data.asfreq('your_freq')` to ensure the index is complete and has a frequency.

### 2. Exogenous Variables (`exog`)
If your model was trained with exogenous variables, you must provide them for the prediction horizon.
*   **Horizon Coverage:** If you call `predict(steps=10)`, your `exog` DataFrame must contain at least 10 

## 7. Include full API reference

When the question needs broad API knowledge, set `include_reference=True` (adds ~7600 tokens).

In [18]:
result = assistant.ask(
    prompt="List all available datasets in skforecast and their frequencies",
    skills=["complete-api-reference"],
    include_reference=True,
)
print(result.explanation)

Skforecast includes over 30 built-in datasets for testing and learning. These cover a wide range of frequencies and use cases, including single-series, multi-series, and exogenous variables.

### Featured Datasets and Frequencies
The most commonly used datasets highlighted in the library include:

| Dataset Name | Frequency | Type |
| :--- | :--- | :--- |
| `h2o` | Monthly | Single series |
| `h2o_exog` | Monthly | Single series + Exogenous |
| `bike_sharing` | Hourly | Single series + Exogenous |
| `items_sales` | Daily | Multi-series (3 series) |
| `store_sales` | Daily | Multi-series (500 series) |
| `fuel_consumption` | Monthly | Single series |
| `australia_electricity` | Hourly | Multi-series |
| `uk_daily_pv` | Daily | Single series |

### Frequency Range
The library provides data across various temporal resolutions, including:
*   **Intraday:** 15 minutes, 30 minutes, and hourly.
*   **Daily:** Standard daily observations.
*   **Aggregated:** Monthly and quarterly series.

### 

## 8. Multi-series explain mode

The assistant detects multi-series data and loads relevant skills automatically.

In [19]:
# Multi-series long-format data
n_multi = 200
dates_multi = pd.date_range("2022-01-01", periods=n_multi, freq="D")
df_multi = pd.DataFrame({
    "date": np.tile(dates_multi, 3),
    "store": ["A"] * n_multi + ["B"] * n_multi + ["C"] * n_multi,
    "revenue": np.concatenate([
        100 + np.cumsum(rng.normal(0, 1, n_multi)),
        200 + np.cumsum(rng.normal(0, 1.5, n_multi)),
        150 + np.cumsum(rng.normal(0, 0.8, n_multi)),
    ]),
})

result = assistant.ask(
    prompt="What approach will you use for these multiple store series?",
    data=df_multi,
    target="revenue",
    date_column="date",
    series_id_column="store",
    steps=7,
)

print(f"Task type: {result.profile.task_type}")
print(f"Forecaster: {result.profile.forecaster}")
print(f"\n{result.explanation}")

Task type: multi_series
Forecaster: ForecasterRecursiveMultiSeries

For this multi-series revenue forecasting task, the approach focuses on training a single "global" model that learns patterns across all three store series simultaneously.

### Forecasting Strategy
We will use **ForecasterRecursiveMultiSeries**. This approach is highly efficient for multiple time series because it trains one shared model (a LightGBM regressor) on all available data. Instead of looking at each store in isolation, the model identifies common trends and behaviors across all series, which often leads to more robust predictions—especially when individual series are short. 

The strategy is **recursive**, meaning the model predicts one day at a time, using its own previous predictions as inputs for the following days until the 7-day horizon is completed.

### Feature Engineering
The model will use two types of historical information to make its predictions:
*   **Lags**: Specific historical points are select

## 9. Results mode — Ask about forecast predictions

After running `forecast()`, pass the `RunResult` to `ask()` so the LLM can explain actual predictions, metrics, and intervals.

In [20]:
# Run a forecast first
forecast_result = assistant.forecast(
    data=df,
    target="sales",
    date_column="date",
    steps=14,
)

print(f"Predictions shape: {forecast_result.predictions.shape}")
print(f"Metrics:\n{forecast_result.metrics.to_string(index=False)}")
print(f"\nPredictions:\n{forecast_result.predictions}")


Predictions shape: (14, 1)
Metrics:
series      MAE      MSE     MASE     MAPE
 sales 0.859057 1.118199 0.300112 0.009848

Predictions:
                 pred
2022-10-20  85.738240
2022-10-21  85.611453
2022-10-22  88.351413
2022-10-23  93.152032
2022-10-24  94.301703
2022-10-25  91.164140
2022-10-26  86.611982
2022-10-27  84.629980
2022-10-28  83.968107
2022-10-29  87.719652
2022-10-30  93.398525
2022-10-31  94.460990
2022-11-01  92.090153
2022-11-02  86.346873


In [21]:
# Ask the LLM about the forecast results
answer = assistant.ask(
    prompt="Explain the predictions. What trend do you see and what values are expected?",
    forecast_result=forecast_result,
)
print(answer.explanation)


Based on the pre-computed forecasting plan and the results provided, here is an explanation of the predicted sales for the 14-day horizon.

### Forecast Overview
The model uses a **recursive strategy** with a LightGBM regressor. It looks at specific historical points (lags) and broader trends (7-day and 91-day averages) while accounting for the "promo" variable. The predictions cover a two-week window from October 20, 2022, to November 2, 2022.

### Observed Trends and Patterns
The forecast reveals a distinct **weekly seasonality**:
*   **Peak Period:** Sales are expected to reach their highest levels early in the week, specifically on **Sundays and Mondays** (e.g., Oct 23–24 and Oct 30–31), with values climbing above 93 units.
*   **Trough Period:** Sales are projected to dip toward the end of the week, hitting their lowest points on **Thursdays and Fridays** (e.g., Oct 27–28), where values drop to approximately 84 units.
*   **Stability:** The pattern repeats almost identically acros

In [22]:
# The result also carries the deterministic code from the forecast
print("Code available:", answer.code is not None)
print(f"Code length: {len(answer.code)} chars")
print(f"\nFirst 5 lines:\n{chr(10).join(answer.code.splitlines()[:5])}")


Code available: True
Code length: 2146 chars

First 5 lines:
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from skforecast.metrics import mean_absolute_scaled_error
from lightgbm import LGBMRegressor
from skforecast.preprocessing import RollingFeatures
